# Baseline Replication: L2P (CVPR 2022) — M0 Linear vs M1 KAC

## 📌 Experiment Overview
- **Target Model:** Learning to Prompt (L2P) (Wang et al., CVPR 2022)
- **Evaluation Dataset:** CUB-200-2011 (200 fine-grained bird species, 10 tasks x 20 classes)
- **Backbone:** Pre-trained Vision Transformer (`ViT-B/16` frozen, 768-dim)
- **Objective:** Strict scientific replication of original baseline (M0) versus naive KAC classification (M1).

## 🔬 Head Configurations (M0 vs M1)
| Mode | Classifier Head | Formulation | Purpose |
| :--- | :--- | :--- | :--- |
| **M0 (Paper Baseline)** | Linear / Cosine Head | $y = \text{Linear}(f)$ or $\text{Norm}(f) \mathbf{W}^\top$ | Replicate original Learning to Prompt (L2P) paper performance |
| **M1 (Naive KAC)** | Kolmogorov-Arnold Head | $y = \text{SplineLinear}(\text{RBF}(\text{LayerNorm}(f)))$ | Test naive non-linear spline classification hypothesis |

## ⚙️ Experimental Controls (Apples-to-Apples)
- **Fixed Seeds:** [0] (with multi-split: 5T, 10T, 20T, 40T)
- **Optimizer:** Adam, standard learning rates
- **Batch Size & Epochs:** `batch_size=128`, `epochs=20`
- **Changelog:** Configured strictly for L2P with MODEL_NAME='L2P'. Evaluates M0 (Linear) vs M1 (KAC LayerNorm + Gaussian RBF + SplineLinear).


In [ ]:
# 1. Điều chỉnh Mô hình và Hyperparameters
MODEL_NAME = 'L2P'  # Chọn 1 trong: 'L2P', 'DualPrompt', 'CODA'

DATASET = 'CUB'
NUM_TASKS = 10
NUM_CLASSES = 200
FIRST_SPLIT = 20
OTHER_SPLIT = 20

BATCH_SIZE = 128
EPOCHS_PER_TASK = 20  # [CUB200-MIGRATION] Default 20 epochs for CUB
LR = 0.001
OPTIMIZER = 'Adam'
SCHEDULE_TYPE = 'cosine'

if MODEL_NAME == 'L2P':
    P_PARAM = [NUM_TASKS, [30, 20, -1]]
elif MODEL_NAME == 'DualPrompt':
    P_PARAM = [NUM_TASKS, [NUM_TASKS, 20, 6]]
elif MODEL_NAME == 'CODA':
    P_PARAM = [NUM_TASKS, [100, 8, 0.0]]
else:
    raise ValueError("Invalid MODEL_NAME")

tasks_logits = [list(range(i * OTHER_SPLIT, (i + 1) * OTHER_SPLIT)) for i in range(NUM_TASKS)]

# 2. Setup Môi trường & Dataset

In [ ]:
import os, shutil, sys
import subprocess

if os.path.exists('KAC'):
    shutil.rmtree('KAC')
subprocess.run(['git', 'clone', 'https://github.com/Ethanhuhuhu/KAC.git'])
subprocess.run(['pip', 'install', 'timm', 'torch', 'torchvision', '--quiet'])

DEST_ROOT = f'KAC/data/{DATASET.lower()}'
os.makedirs(DEST_ROOT, exist_ok=True)

def find_ready_dataset(base_dir='/kaggle/input'):
    for root, dirs, files in os.walk(base_dir):
        if 'train' in dirs and 'test' in dirs:
            if os.path.exists(os.path.join(root, 'train', '001.Black_footed_Albatross')):
                return os.path.join(root, 'train'), os.path.join(root, 'test')
    return None, None

found_train, found_test = find_ready_dataset()
if found_train and found_test:
    print(f"Dataset tìm thấy tại: {os.path.dirname(found_train)}")
    if not os.path.exists(os.path.join(DEST_ROOT, 'train')): os.symlink(found_train, os.path.join(DEST_ROOT, 'train'))
    if not os.path.exists(os.path.join(DEST_ROOT, 'test')): os.symlink(found_test, os.path.join(DEST_ROOT, 'test'))
    print("Symlink Dataset Sẵn sàng!")
else:
    print("CẢNH BÁO: Không tìm thấy dataset train/test đã chia sẵn!")

if not os.path.exists(f'data/{DATASET.lower()}') and os.path.exists(DEST_ROOT):
    os.makedirs('data', exist_ok=True)
    os.symlink(os.path.abspath(DEST_ROOT), f'data/{DATASET.lower()}')

# 3. KAC Patcher (Minimalist & Safe)
Đã dỡ bỏ toàn bộ code Gradient Accumulation dư thừa. 
Chỉ giữ lại **ĐÚNG 100%** những patch an toàn từ file `l2p_kac_cub200.ipynb` gốc của bạn.

In [ ]:
print("Applying Minimal & Safe Patches...")

# --- PATCH 1: models/zoo.py ---
zoo_path = 'KAC/models/zoo.py'
zoo_text = open(zoo_path).read()
zoo_text = zoo_text.replace('prompt_param=prompt_para', 'prompt_param=prompt_param')
zoo_text = zoo_text.replace('KACLayer(768, num_classes)', 'KACLayer(768, num_classes, grid_min=-2., grid_max=2., num_grids=4)')
open(zoo_path, 'w').write(zoo_text)

# --- PATCH 2: learners/prompt.py ---
prompt_path = 'KAC/learners/prompt.py'
prompt_text = open(prompt_path).read()

# Fix 1: Chèn clf_type để KAC có thể hoạt động
prompt_text = prompt_text.replace("prompt_param=self.prompt_param)", 
                                  "prompt_param=self.prompt_param, clf_type=cfg.get('clf_type', 'linear'))")

# Fix 2: Khởi tạo AMP Scaler trong __init__
old_init = "self.prompt_param = learner_config['prompt_param']\n        super(Prompt, self).__init__(learner_config)"
new_init = "self.prompt_param = learner_config['prompt_param']\n        super(Prompt, self).__init__(learner_config)\n        self.scaler = torch.cuda.amp.GradScaler()\n        for p in self.model.feat.parameters(): p.requires_grad = False"
prompt_text = prompt_text.replace(old_init, new_init)

# Fix 3: Thêm Autocast vào forward pass
prompt_text = prompt_text.replace('logits, prompt_loss = self.model(inputs, train=True)', 
                                  'with torch.cuda.amp.autocast():\n            logits, prompt_loss = self.model(inputs, train=True)')

# Fix 4: Sửa hàm update_model thành bản chuẩn, gọn nhẹ, có ghi log gradient (không có Accumulation rườm rà)
old_update = """        # ce loss
        total_loss = total_loss + prompt_loss.sum()

        # step
        self.optimizer.zero_grad()
        total_loss.backward()
        self.optimizer.step()"""

new_update = """        # ce loss
        total_loss = total_loss + prompt_loss.sum()

        # step (AMP + Grad Log)
        self.optimizer.zero_grad()
        self.scaler.scale(total_loss).backward()
        self.scaler.unscale_(self.optimizer)
        
        if not hasattr(self, 'grad_logs'): self.grad_logs = []
        if not hasattr(self, '_batch_step'): self._batch_step = 0
        self._batch_step += 1
        if self._batch_step % 19 == 0:
            try:
                m = self.model
                p_grad = sum(p.grad.norm().item()**2 for p in m.prompt.parameters() if p.grad is not None)**0.5 if hasattr(m, 'prompt') else 0.0
                c_grad = sum(p.grad.norm().item()**2 for p in m.last.parameters() if p.grad is not None)**0.5 if hasattr(m, 'last') else 0.0
                self.grad_logs.append({'step': self._batch_step, 'p_grad': round(p_grad, 4), 'c_grad': round(c_grad, 4)})
            except: pass

        self.scaler.step(self.optimizer)
        self.scaler.update()"""
prompt_text = prompt_text.replace(old_update, new_update)
open(prompt_path, 'w').write(prompt_text)

# --- PATCH 3: learners/default.py ---
def_path = 'KAC/learners/default.py'
text_def = open(def_path).read()
text_def = text_def.replace('model.eval()\n        for i,', 'model.eval()\n        torch.set_grad_enabled(False)\n        for i,')
text_def = text_def.replace('model.train(orig_mode)', 'torch.set_grad_enabled(True)\n        model.train(orig_mode)')
text_def = text_def.replace("self.model = torch.nn.DataParallel(self.model, device_ids=self.config['gpuid'], output_device=self.config['gpuid'][0])", "self.model = self.model")
open(def_path, 'w').write(text_def)

print("All Minimal Patches Applied Successfully!")

# 4. Training Loop và Đánh giá

In [ ]:
sys.path.append('KAC')
import torch
import json
import time
import os
from torch.utils.data import DataLoader

import learners
import random
import numpy as np
import dataloaders

# =====================================================================
# CẤU HÌNH CÁC KỊCH BẢN THÍ NGHIỆM TỰ ĐỘNG
# =====================================================================
experiments = [
    {"name": "5T_40C",  "num_tasks": 5, "num_classes": 200, "split": 40},
    {"name": "10T_20C", "num_tasks": 10, "num_classes": 200, "split": 20},
    {"name": "20T_10C", "num_tasks": 20, "num_classes": 200, "split": 10},
    {"name": "40T_5C",  "num_tasks": 40, "num_classes": 200, "split": 5}
]

if MODEL_NAME == 'L2P':
    LearnerClass = learners.prompt.L2P
elif MODEL_NAME == 'DualPrompt':
    LearnerClass = learners.prompt.DualPrompt
elif MODEL_NAME == 'CODA':
    LearnerClass = learners.prompt.CODAPrompt

results = {}
if os.path.exists('/kaggle/working/results.json'):
    print("Found existing results.json, loading...")
    with open('/kaggle/working/results.json', 'r') as f:
        results = json.load(f)

for exp in experiments:
    _NUM_TASKS = exp["num_tasks"]
    _NUM_CLASSES = exp["num_classes"]
    _SPLIT = exp["split"]
    
    # 1. Khởi tạo lại Dataset cho config này
    seed = 0
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # LUÔN LUÔN xáo trộn trên bộ 200 class gốc để giữ nguyên vẹn thứ tự với chung 1 seed
    class_order = np.arange(NUM_CLASSES).tolist()  # NUM_CLASSES = 200 (Global)
    random.shuffle(class_order)
    
    # Chia theo split, sau đó CHỈ LẤY đúng số lượng Task yêu cầu (cắt lát - slicing)
    all_tasks = [class_order[p:p+_SPLIT] for p in range(0, NUM_CLASSES, _SPLIT)]
    tasks = all_tasks[:_NUM_TASKS]
    
    # Cập nhật lại số lượng class THỰC TẾ sẽ được train trong kịch bản này
    _REAL_NUM_CLASSES = _NUM_TASKS * _SPLIT
    
    train_tf = dataloaders.utils.get_transform(dataset='CUB', phase='train', aug=True, resize_imnet=True)
    test_tf  = dataloaders.utils.get_transform(dataset='CUB', phase='test',  aug=True, resize_imnet=True)
    
    train_dataset = dataloaders.iCUB('./data', train=True,  lab=True, tasks=tasks, download_flag=False, transform=train_tf, seed=seed, rand_split=True)
    test_dataset  = dataloaders.iCUB('./data', train=False, tasks=tasks, download_flag=False, transform=test_tf,  seed=seed, rand_split=True)
    
    # Dynamic P_PARAM mapping
    if MODEL_NAME == 'L2P':
        _P_PARAM = [_NUM_TASKS, [30, 20, -1]]
    elif MODEL_NAME == 'DualPrompt':
        _P_PARAM = [_NUM_TASKS, [_NUM_TASKS, 20, 6]]
    elif MODEL_NAME == 'CODA':
        _P_PARAM = [_NUM_TASKS, [100, 8, 0.0]]
        
    for clf_type in ['linear', 'kac']:
        run_name = f"{MODEL_NAME}_{clf_type}_{exp['name']}"
        print(f"\n{'='*60}\n BẮT ĐẦU: {MODEL_NAME} + {clf_type.upper()} | Config: {exp['name']}\n{'='*60}")
        
        cfg = {
            'num_classes': NUM_CLASSES,
            'lr': LR,
            'optimizer': OPTIMIZER,
            'schedule': [EPOCHS_PER_TASK],
            'schedule_type': SCHEDULE_TYPE,
            'batch_size': BATCH_SIZE,
            'model_type': 'zoo',
            'model_name': 'vit_pt_imnet',
            'gpuid': [0],
            'memory': 0,
            'out_dim': NUM_CLASSES,
            'overwrite': True,
            'DW': False,
            'top_k': 1,
            'dil': False,
            'clf_type': clf_type,
            'tasks': tasks,
            'prompt_param': _P_PARAM,
            'weight_decay': 0.0,
            'momentum': 0.9,
            'debug_mode': False,
            'temp': 2.0,
            'upper_bound_flag': False
        }

        learner = LearnerClass(cfg)
        learner.cuda()
        
        start_task = 0
        for t in range(_NUM_TASKS - 1, -1, -1):
            ckpt_path = f"/kaggle/working/ckpt_{run_name}_task{t}.pt"
            if os.path.exists(ckpt_path):
                print(f"Resuming from task {t+1}...")
                for i in range(t + 1):
                    learner.task_count = i
                    learner.add_valid_output_dim(_SPLIT)
                    if i > 0: learner.model.prompt.process_task_count()
                learner.last_valid_out_dim = learner.valid_out_dim
                state = torch.load(ckpt_path)
                learner.model.load_state_dict(state['model_state_dict'])
                learner.optimizer.load_state_dict(state['optimizer_state_dict'])
                start_task = t + 1
                break
                
        if run_name not in results:
            results[run_name] = {}
            
        for t in range(start_task, _NUM_TASKS):
            print(f"\n--- [ {exp['name']} ] Task {t+1}/{_NUM_TASKS} ---")
            try:
                learner.model.module.task_id = t
            except:
                learner.model.task_id = t

            learner.task_count = t
            learner.add_valid_output_dim(_SPLIT)
            if t > 0: learner.model.prompt.process_task_count()
                    
            train_dataset.load_dataset(t, train=True)
            train_dataset.append_coreset(only=False)
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2)
            
            test_dataset.load_dataset(t, train=False)
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=2)
            
            if not os.path.exists("/kaggle/working/temp_model"): 
                os.makedirs("/kaggle/working/temp_model")
            
            # Reset grad logs before learning
            if hasattr(learner, 'grad_logs'): learner.grad_logs = []
            if hasattr(learner, '_batch_step'): learner._batch_step = 0

            learner.learn_batch(train_loader, train_dataset, "/kaggle/working/temp_model", test_loader)
            
            task_grads = getattr(learner, 'grad_logs', [])
            
            accs = []
            for j in range(t + 1):
                test_dataset.load_dataset(j, train=True)
                task_test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
                acc = learner.validation(task_test_loader)
                accs.append(acc)
                
            results[run_name][str(t)] = {"accs": accs, "grads": task_grads}
            
            with open('/kaggle/working/results.json', 'w') as f:
                json.dump(results, f, indent=4)
            
            # Lưu checkpoint hiện tại
            ckpt_path = f"/kaggle/working/ckpt_{run_name}_task{t}.pt"
            torch.save({
                'model_state_dict': learner.model.state_dict(),
                'optimizer_state_dict': learner.optimizer.state_dict(),
                'task_index': t,
                'valid_out_dim': learner.valid_out_dim,
            }, ckpt_path)
            
            # XÓA checkpoint cũ để giữ dung lượng < 20GB
            if t > 0:
                prev_ckpt = f"/kaggle/working/ckpt_{run_name}_task{t-1}.pt"
                if os.path.exists(prev_ckpt):
                    os.remove(prev_ckpt)

# 5. In bảng Report (Kịt bản Metric gốc của Forgetting KAC)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

summary = {}
waypoints = [5, 10, 20, 40]

for run_name, tasks_data in results.items():
    n = len(tasks_data)
    if n == 0: continue
    
    # Tạo ma trận vuông T x T
    R = np.zeros((n, n))
    for t_idx in range(n):
        t_str = str(t_idx)
        if t_str in tasks_data:
            accs = tasks_data[t_str]['accs']
            for j_idx, acc in enumerate(accs):
                R[t_idx, j_idx] = acc
    
    summary[run_name] = {'step_avgs': [np.mean(R[i, :i+1]) for i in range(n)]}
    
    # --- TÍNH METRIC TẠI CÁC CỘT MỐC (WAYPOINTS) ---
    for k in waypoints:
        if k <= n:
            R_k = R[:k, :k]
            
            final_avg = np.mean(R_k[k-1, :])
            last_acc = R_k[k-1, k-1]
            
            fgt = 0.0
            if k > 1:
                for step in range(1, k):
                    step_drop = 0.0
                    for i in range(step):
                        step_drop += (R_k[step-1, i] - R_k[step, i])
                    fgt += (step_drop / step)
                forgetting = fgt / (k - 1)
            else:
                forgetting = 0.0
                
            summary[run_name][f'WP_{k}'] = {
                'Avg': final_avg,
                'Last': last_acc,
                'FG': forgetting
            }

# --- IN BẢNG BÁO CÁO NHIỀU CỘT MỐC ---
print(f"\n{'='*140}")
print(f" BÁO CÁO WAYPOINT (5-10-20-40 TASKS): {MODEL_NAME} ({DATASET}) ")
print(f"{'='*140}")

# Header
header = f"| {'Run Name':<30} |"
for k in waypoints:
    header += f" WP {k} (Avg|Last|FG) |"
print(header)
print("|" + "-"*32 + ("|" + "-"*20)*len(waypoints) + "|")

# Data
for run_name, data in summary.items():
    row = f"| {run_name:<30} |"
    for k in waypoints:
        if f'WP_{k}' in data:
            mets = data[f'WP_{k}']
            row += f" {mets['Avg']:>4.1f}|{mets['Last']:>4.1f}|{mets['FG']:>4.1f} |"
        else:
            row += f" {'-':>18} |"
    print(row)
print("\n")

# --- VẼ BIỂU ĐỒ ---
plt.figure(figsize=(12, 8))
for run_name, data in summary.items():
    marker = 's' if 'kac' in run_name else 'o'
    plt.plot(range(1, len(data['step_avgs']) + 1), data['step_avgs'], marker=marker, label=run_name)

# Đánh dấu các cột mốc
for k in waypoints:
    plt.axvline(x=k, color='gray', linestyle='--', alpha=0.5)

plt.xlabel('Task Index')
plt.ylabel('Average Incremental Accuracy (%)')
plt.title(f'{DATASET} Incremental Accuracy: {MODEL_NAME} Auto-Suite')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.savefig(f'/kaggle/working/{MODEL_NAME}_{DATASET}_auto_suite_plot.png')
plt.show()

In [ ]:
# ==============================================================================
# Ô CODE TÍNH TOÁN TÀI NGUYÊN (Chạy SAU KHI train xong)
# ==============================================================================
!pip install thop -q
import torch
import time
from thop import profile

def profile_model(learner_cfg, learner_class):
    print(f"\n{'='*50}\n BÁO CÁO TÀI NGUYÊN: {MODEL_NAME}\n{'='*50}")
    
    # Khởi tạo một phiên bản ảo của mô hình để đo lường
    learner = learner_class(learner_cfg)
    learner.cuda()
    learner.task_count = 9  # Giả lập đang ở Task cuối cùng (10)
    learner.add_valid_output_dim(20)
    
    m = learner.model.module if hasattr(learner.model, 'module') else learner.model
    m.eval()

    # 1. ĐO LƯỜNG SỐ LƯỢNG THAM SỐ
    total_params = sum(p.numel() for p in m.parameters())
    train_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"Tổng tham số (Total Params):      {total_params / 1e6:.2f} M")
    print(f"Tham số học (Trainable Params):   {train_params / 1e6:.2f} M")

    # 2. ĐO LƯỜNG GFLOPs (Sử dụng wrapper để tương thích thư viện THOP)
    class Wrapper(torch.nn.Module):
        def __init__(self, core_model):
            super().__init__()
            self.core_model = core_model
        def forward(self, x):
            # Ép mô hình chạy luồng Test
            self.core_model.task_id = 9
            return self.core_model(x, train=False)
    
    wrapper = Wrapper(m).cuda()
    dummy_input = torch.randn(1, 3, 224, 224).cuda()
    
    try:
        macs, _ = profile(wrapper, inputs=(dummy_input, ), verbose=False)
        print(f"Khối lượng tính toán (GFLOPs):    {macs / 1e9:.2f} GFLOPs")
    except Exception as e:
        print("Không thể đo GFLOPs:", e)

    # 3. ĐO LƯỜNG THỜI GIAN SUY LUẬN (INFERENCE TIME)
    print("\nĐang đo lường Inference Time (Batch Size = 1) trên GPU...")
    with torch.no_grad():
        # Warm-up (Khởi động GPU để tránh lag nhịp đầu)
        for _ in range(50):
            wrapper(dummy_input)
            
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)
        
        # Bắt đầu bấm giờ đồng bộ 1000 lượt suy luận
        start_event.record()
        for _ in range(1000):
            wrapper(dummy_input)
        end_event.record()
        
        torch.cuda.synchronize() # Ép CPU đợi GPU chạy xong
        
        avg_time_ms = start_event.elapsed_time(end_event) / 1000.0
        fps = 1000.0 / avg_time_ms
        print(f"Thời gian suy luận (Latency):     {avg_time_ms:.2f} ms / ảnh")
        print(f"Tốc độ xử lý (Throughput):        {fps:.2f} ảnh / giây")
    print("="*50)

# Kích hoạt hàm đo lường
profile_model(cfg, LearnerClass)